#Loading Dataset

##Mount Google Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


##Set required paths

In [3]:
dataset_path = '/content/drive/MyDrive/1:1_Arjan_Walia/Vessels/datasets/25791978'
save_folder = '/content/drive/MyDrive/1:1_Arjan_Walia/Vessels/datasets/processed_files'
models = '/content/drive/MyDrive/1:1_Arjan_Walia/Vessels/models'

#Extracting Features

##Function to Extract Audio Features

In [5]:
import os
import pandas as pd
import numpy as np
import librosa

def extract_acoustic_features(file_path):
    """Extracts MFCCs and spectral features from a Wolfset recording."""
    try:
        # Load audio - 10 seconds is enough for a stable motor signature
        y, sr = librosa.load(file_path, duration=10.0)

        # 1. Apply 50Hz Notch Filter (removes electrical hum mentioned in paper)
        # Simplified here using a high-pass filter for the 'simplest' approach
        y = librosa.effects.preemphasis(y)

        # 2. Extract 13 MFCCs (Standard for acoustic classification)
        mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
        mfccs_mean = np.mean(mfccs.T, axis=0)

        # 3. Extract Spectral Rolloff (helps distinguish engine 'brightness')
        rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr)
        rolloff_mean = np.mean(rolloff)

        return np.hstack([mfccs_mean, rolloff_mean])
    except Exception as e:
        return None

##Process Files

In [17]:
from tqdm.notebook import tqdm

# 2. Process all files based on the Wolfset naming convention
dataset = []
files = [file_ for file_ in os.listdir(dataset_path) if file_.endswith('.wav')]

for audio_file in tqdm(files):
    # Extract labels from filename template: XxxxxxTttNnn
    # Motor digits 1-4 are combustion, digit 5 is electric [cite: 153]
    motor_code = audio_file[1:6]
    transient_code = audio_file[7:9]
    noise_code = audio_file[10:12]

    # Determine the "Primary Motor" for this file
    # We label based on which motor is NOT '0' (Absent) [cite: 155, 171]
    primary_motor = None
    for i, val in enumerate(motor_code):
        if val != '0':
            primary_motor = i + 1 # Motors 1-5
            break

    if primary_motor is not None:
        file_path = os.path.join(dataset_path, audio_file)
        features = extract_acoustic_features(file_path)

        if features is not None:
            row = list(features) + [primary_motor, motor_code, transient_code, noise_code, audio_file]
            dataset.append(row)

# 3. Save to CSV
columns = [f'mfcc_{index}' for index in range(13)] + ['spectral_rolloff', 'label', 'motor_code', 'transient_code', 'noise_code', 'filename']
df = pd.DataFrame(dataset, columns=columns)
print(len(dataset))

  0%|          | 0/168 [00:00<?, ?it/s]

122


##Pre-processing

In [7]:
COLUMNS_EXCLUDE = ['label', 'motor_code', 'transient_code', 'noise_code', 'filename']

In [8]:
from sklearn.model_selection import train_test_split

# splitting
train_df, test_df = train_test_split(df, test_size = 0.3, stratify = df["label"], random_state = 42)
# reset index
train_df.reset_index(drop = True, inplace = True)
test_df.reset_index(drop = True, inplace = True)

In [9]:
import pickle

def preprocessing_model_saver(model, path_name):
    try:
        with open(path_name, "wb") as model_file:
            pickle.dump(model, model_file)

        return True
    except Exception as error:
        print("Error generated while saving the model: {}".format(str(error)))

        return False

In [10]:
from sklearn.preprocessing import StandardScaler

columns_to_scale = [cols for cols in list(train_df.columns) if cols not in COLUMNS_EXCLUDE]
print(columns_to_scale)

# fit scale
stc = StandardScaler().fit(train_df[columns_to_scale])

train_df[columns_to_scale] = stc.transform(train_df[columns_to_scale])
test_df[columns_to_scale] = stc.transform(test_df[columns_to_scale])

# save the model
model_path = os.path.join(models, "scaler")
preprocessing_model_saver(stc, model_path)

['mfcc_0', 'mfcc_1', 'mfcc_2', 'mfcc_3', 'mfcc_4', 'mfcc_5', 'mfcc_6', 'mfcc_7', 'mfcc_8', 'mfcc_9', 'mfcc_10', 'mfcc_11', 'mfcc_12', 'spectral_rolloff']


True

In [11]:
# save train
train_path = os.path.join(save_folder, "wolfset_train.csv")
train_df.to_csv(train_path, index = False)

# save test
test_path = os.path.join(save_folder, "wolfset_test.csv")
test_df.to_csv(test_path, index = False)

In [12]:
import os
import pandas as pd
train_path = os.path.join(save_folder, "wolfset_train.csv")
train_df = pd.read_csv(train_path)

train_df["label"].value_counts()

,count
label,
1,40
2,13
3,13
4,11
5,8


In [13]:
test_path = os.path.join(save_folder, "wolfset_test.csv")
test_df = pd.read_csv(test_path)


In [14]:
import os
import pandas as pd
train_path = os.path.join(save_folder, "wolfset_test.csv")
train_df = pd.read_csv(test_path)

test_df["label"].value_counts()

,count
label,
1,17
2,6
4,5
3,5
5,4


In [16]:
from pathlib import Path

dir = Path('/content/drive/MyDrive/1:1_Arjan_Walia/Vessels/datasets/25791978')
i = 0
for file in dir.glob('*.wav'):
  i+=1

print(i)


168
